In [1]:
import numpy as np
import pandas as pd
import os
import glob
import rasterio
from rasterio.transform import from_origin
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
import rioxarray as rxr
import xarray as xr
import geostatspy.geostats as geostats
import geostatspy.GSLIB as GSLIB
from pyproj import Proj

# ── Constants ──────────────────────────────────────────────────────────────
GSLIB_SENTINEL = 9.966e+20    
UTM_PROJ       = "+proj=utm +zone=45 +ellps=WGS84 +datum=WGS84 +units=m +no_defs"
NODATA_OUT     = -9999.0
GRID_RES_KM    = 1.0          

seasons = {
    'winter':       [12, 1, 2],
    'pre_monsoon':  [3, 4, 5],
    'monsoon':      [6, 7, 8, 9],
    'post_monsoon': [10, 11]
}
model_it    = {'spherical': 1, 'exponential': 2, 'gaussian': 3}
tmin, tmax  = -9999, 9999

# ── Paths ──────────────────────────────────────────────────────────────────
era5L_dir    = r"..\Data\Datasets\ERA5L_prec"
var_dir      = r"..\Data\Processed\Variogram_analysis"
var_params_path = r"..\Data\Processed\Variogram_analysis\variogram_params_singleDomain.csv"
temp_dir     = r"..\Data\Processed\Kriging_Predictors\Temp"
output_dir   = r"..\Data\Processed\Kriging_Predictors\Kriged_rasters"
os.makedirs(temp_dir,   exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

In [2]:
# ── Helper functions ───────────────────────────────────────────────────────
def get_season(month):
    for s, months in seasons.items():
        if month in months:
            return s
    return None

def coords_to_utm_km(lons, lats):
    proj = Proj(UTM_PROJ)
    x_m, y_m = proj(lons, lats)
    return x_m / 1000.0, y_m / 1000.0

def apply_trend(df, vp, xcol='x_km', ycol='y_km'):
    return (
        vp.coef_x  * df[xcol]           +
        vp.coef_y  * df[ycol]           +
        vp.coef_x2 * df[xcol] ** 2      +
        vp.coef_y2 * df[ycol] ** 2      +
        vp.coef_xy * df[xcol] * df[ycol]+
        vp.intercept
    )

def safe_nscore_lookup(ns_pred, lookup_df):
    lut = lookup_df.sort_values('nscore_value').drop_duplicates('nscore_value')
    return np.interp(ns_pred, lut['nscore_value'].values, lut['original_value'].values)

def mask_gslib_sentinel(arr, threshold=1e10):
    return np.where(np.abs(arr) > threshold, np.nan, arr.astype(float))

# ── Pre-load logic ────────────────────────────────────────────────────────
era5L_files = sorted(glob.glob(os.path.join(era5L_dir, "*.tif")))
era5L_dates = [os.path.basename(f)[-12:-4] for f in era5L_files]

# Setup grid using a sample ERA5-L file
with rxr.open_rasterio(era5L_files[0]) as sample:
    sample_grid = sample.squeeze('band', drop=True)
    lon_grid, lat_grid = np.meshgrid(sample_grid.x.values, sample_grid.y.values)
    x_km_all, y_km_all = coords_to_utm_km(lon_grid.ravel(), lat_grid.ravel())

# Load single domain variogram parameters
var_params_all = pd.read_csv(var_params_path).set_index('season')

In [ ]:
for day_idx, date_str in enumerate(era5L_dates):
    month  = int(date_str[4:6])
    season = get_season(month)
    print(f"\n{'-'*60}\n  Date: {date_str}  |  Season: {season}")

    if season not in var_params_all.index:
        print(f"  [SKIP] No params for season {season}"); continue

    with rxr.open_rasterio(era5L_files[day_idx]) as era5_rxr:
        era5_arr = era5_rxr.squeeze('band', drop=True).load().values

    # Base dataframe for the entire domain
    base_df = pd.DataFrame({
        'x_km': x_km_all, 
        'y_km': y_km_all,
        'era5_prec': era5_arr.ravel().astype(float)
    }).dropna(subset=['era5_prec'])
    base_df = base_df[base_df['era5_prec'] >= 0].reset_index(drop=True)

    if len(base_df) < 4:
        print(f"  [SKIP] Only {len(base_df)} pixels"); continue

    vp = var_params_all.loc[season].copy()

    # 1. Detrending
    if vp.trend_removed:
        base_df['residual'] = base_df['era5_prec'] - apply_trend(base_df, vp)
    else:
        base_df['residual'] = base_df['era5_prec']

    if base_df['residual'].std() < 1e-6:
        print("  [SKIP] Zero variance"); continue

    # 2. N-score
    ns_data, t_vals, t_ns = geostats.nscore(base_df, 'residual')
    base_df['ns'] = ns_data
    lookup_df = pd.DataFrame({'original_value': t_vals, 'nscore_value': t_ns})

    # 3. Kriging Setup
    h_range = vp['range_km']
    vario = GSLIB.make_variogram(
        nug  = vp['nugget'], 
        nst  = 1, 
        it1  = model_it[vp['model_type']], 
        cc1  = vp['partial_sill'], 
        azi1 = vp['azimuth_major'], 
        hmaj1 = h_range, 
        hmin1 = h_range  # Isotropic
    )

    x_min, x_max = base_df['x_km'].min(), base_df['x_km'].max()
    y_min, y_max = base_df['y_km'].min(), base_df['y_km'].max()
    xmn, ymn = np.floor(x_min) + GRID_RES_KM/2, np.floor(y_min) + GRID_RES_KM/2
    nx, ny = int(np.ceil((x_max-xmn)/GRID_RES_KM))+1, int(np.ceil((y_max-ymn)/GRID_RES_KM))+1

    # 4. Krige
    kmap, _ = geostats.kb2d(df=base_df, xcol='x_km', ycol='y_km', vcol='ns', 
                            tmin=tmin, tmax=tmax, nx=nx, xmn=xmn, xsiz=GRID_RES_KM, 
                            ny=ny, ymn=ymn, ysiz=GRID_RES_KM, nxdis=1, nydis=1, 
                            ndmin=1, ndmax=8, radius=float(h_range), ktype=1, skmean=0, vario=vario)

    kmap = mask_gslib_sentinel(kmap)
    if np.isnan(kmap).all():
        print("  [SKIP] No nodes estimated"); continue

    # 5. Back-transform & Trend recovery
    x_nodes = xmn + np.arange(nx) * GRID_RES_KM
    y_nodes = ymn + (ny - 1 - np.arange(ny)) * GRID_RES_KM 
    xx, yy  = np.meshgrid(x_nodes, y_nodes)

    out = pd.DataFrame({'x_km': xx.ravel(), 'y_km': yy.ravel(), 'ns_pred': kmap.ravel()}).dropna()
    out['residual_pred'] = safe_nscore_lookup(out['ns_pred'].values, lookup_df)
    out['trend'] = apply_trend(out, vp) if vp.trend_removed else 0.0
    out['prec_pred'] = (out['residual_pred'] + out['trend']).clip(lower=0)

    # 6. Rasterize & Save
    all_x, all_y = np.sort(out['x_km'].unique()), np.sort(out['y_km'].unique())[::-1]
    grid_pivot = out.pivot(index='y_km', columns='x_km', values='prec_pred').reindex(index=all_y, columns=all_x)
    out_grid = np.where(np.isnan(grid_pivot.values), NODATA_OUT, grid_pivot.values).astype('float32')

    transform_utm = from_origin(west=all_x.min()*1000 - (GRID_RES_KM*500), 
                                north=all_y.max()*1000 + (GRID_RES_KM*500), 
                                xsize=GRID_RES_KM*1000, ysize=GRID_RES_KM*1000)

    utm_path = os.path.join(temp_dir, f"utm_{date_str}.tif")
    final_path = os.path.join(output_dir, f"kriged_{date_str}_singleDomain.tif")

    with rasterio.open(utm_path, 'w', driver='GTiff', height=out_grid.shape[0], width=out_grid.shape[1], 
                       count=1, dtype='float32', crs='EPSG:32645', transform=transform_utm, nodata=NODATA_OUT) as dst:
        dst.write(out_grid, 1)

    # Reproject to WGS84
    with rasterio.open(utm_path) as src:
        t_wgs, w, h = calculate_default_transform(src.crs, 'EPSG:4326', src.width, src.height, *src.bounds)
        out_wgs = np.full((1, h, w), NODATA_OUT, dtype='float32')
        reproject(source=rasterio.band(src, 1), destination=out_wgs, src_transform=src.transform, 
                  src_crs=src.crs, dst_transform=t_wgs, dst_crs='EPSG:4326', 
                  resampling=Resampling.bilinear, src_nodata=NODATA_OUT, dst_nodata=NODATA_OUT)
        
        profile = src.profile.copy()
        profile.update(crs='EPSG:4326', transform=t_wgs, width=w, height=h)
        with rasterio.open(final_path, 'w', **profile) as dst:
            dst.write(out_wgs)

    os.remove(utm_path)
    print(f"  [OK] → {final_path}")


------------------------------------------------------------
  Date: 19910101  |  Season: winter
 make_variogram Warning: sill does not sum to 1.0, do not use in simulation 


100%|██████████| 512/512 [00:49<00:00, 10.41it/s]


  Estimated   451584 blocks 
      average   -0.037822466868163276  variance  1.0187827312782163
  [OK] → ..\Data\Processed\Kriging_Predictors\Kriged_rasters\kriged_19910101_singleDomain.tif

------------------------------------------------------------
  Date: 19910102  |  Season: winter
 make_variogram Warning: sill does not sum to 1.0, do not use in simulation 


100%|██████████| 512/512 [01:00<00:00,  8.52it/s]


  Estimated   451584 blocks 
      average   0.025347424079152265  variance  0.7838743749842869
  [OK] → ..\Data\Processed\Kriging_Predictors\Kriged_rasters\kriged_19910102_singleDomain.tif

------------------------------------------------------------
  Date: 19910103  |  Season: winter
 make_variogram Warning: sill does not sum to 1.0, do not use in simulation 


 27%|██▋       | 138/512 [00:18<00:45,  8.27it/s]